<a href="https://colab.research.google.com/github/tradingwithme/Works-w-ML-and-w-o-ML/blob/main/LIU%20Alumni%20Global%20dataset%20merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
import pandas as pd
from google.colab import userdata
API_TOKEN = userdata.get('AIRTABLE_TOKEN')

# List of (Base ID, Table ID) tuples
bases = [
    ("appWpEREOGinuvtlG", "tblXQbkjJszZYobKH"),
    ("appnlmbXz3EcmhPKH", "tblk3Ek7IXJem4yqb"),
    ("apphzRVKvbog9V6Yv", "tbltkFiGqano0tNa2")
]

headers = {
    "Authorization": f"Bearer {API_TOKEN}"
}

all_data = []

for base_id, table_id in bases:
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"

    while url:
        response = requests.get(url, headers=headers)
        data = response.json()

        records = data.get("records", [])
        for r in records:
            all_data.append(r["fields"])

        offset = data.get("offset")
        url = f"https://api.airtable.com/v0/{base_id}/{table_id}?offset={offset}" if offset else None

df = pd.DataFrame(all_data)

In [8]:
df.to_csv("Alumni Survey Results.csv", index=False)

In [9]:
!pip -q install openpyxl python-docx rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.7 MB/s eta 0:00:00


In [11]:
!rm -r sample_data

In [16]:
from os import listdir
[i for i in listdir('/content/') if len(i.split('.'))==2 and all(len(j) for j in i.split('.'))]

['Feb 2026 LIU Global Alumni Data Research(Sheet1).csv',
 'AllGlobalStudents Alumni Soenke(All Global Graduates).csv',
 'LIU Global Informal Gathering in Brooklyn(Sheet 1 - LIU Global Informal G).csv',
 'Alumni Survey Results.csv',
 'ALumni- 5 yrs.docx',
 'LIST OF ALUMNI from Teresa.docx',
 'LIU Global alumni research on Linked-In(in).csv',
 'Emails alums 2020-2024 grads.docx',
 'LIU Global Alumni Contacts May 2024.csv']

In [33]:
import os, re
import pandas as pd

BASE = "/content"

paths = {
  "template": os.path.join(BASE, "Feb 2026 LIU Global Alumni Data Research(Sheet1).csv"),
  "soenke": os.path.join(BASE, "AllGlobalStudents Alumni Soenke(All Global Graduates).csv"),
  "linkedin": os.path.join(BASE, "LIU Global alumni research on Linked-In(in).csv"),
  "gathering": os.path.join(BASE, "LIU Global Informal Gathering in Brooklyn(Sheet 1 - LIU Global Informal G).csv"),
  "survey": os.path.join(BASE, "Alumni Survey Results.csv"),
  "contacts2024": os.path.join(BASE, "LIU Global Alumni Contacts May 2024.csv"),
  "docx_5yrs": os.path.join(BASE, "ALumni- 5 yrs.docx"),
  "docx_teresa": os.path.join(BASE, "LIST OF ALUMNI from Teresa.docx"),
  "docx_emails": os.path.join(BASE, "Emails alums 2020-2024 grads.docx"),
}

missing = [k for k,v in paths.items() if not os.path.exists(v)]
if missing:
    raise FileNotFoundError(f"Missing files for keys: {missing}\nCheck filenames in /content/")
print("All files found.")
from rapidfuzz import fuzz

def clean_str(x):
    if pd.isna(x): return ""
    return str(x).strip()

def norm_name(s):
    s = clean_str(s).lower()
    s = re.sub(r"[\u2019']", "", s)
    s = re.sub(r"[^a-z\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_last_first(name):
    name = clean_str(name)
    if "," in name:
        last, first = [p.strip() for p in name.split(",", 1)]
        return first, last
    parts = re.split(r"\s+", name.strip())
    if len(parts) >= 2:
        return parts[0], parts[-1]
    return name, ""

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(\+?\d[\d\-\(\)\s/]{7,}\d)")

def extract_emails(text):
    return sorted(set(EMAIL_RE.findall(clean_str(text))))

def extract_phones(text):
    raw = PHONE_RE.findall(clean_str(text))
    out = []
    for p in raw:
        out.append(re.sub(r"\s+", " ", p).strip())
    return sorted(set(out))

def append_prev(existing, new_value, sep="; "):
    existing = clean_str(existing)
    new_value = clean_str(new_value)
    if not new_value:
        return existing
    if not existing:
        return new_value
    if new_value.lower() in existing.lower():
        return existing
    return existing + sep + new_value

# Target columns (taken from the Feb 2026 template file)
TARGET_COLS = list(pd.read_csv(paths["template"]).columns)

def empty_target_row():
    return {c: "" for c in TARGET_COLS}

def safe_get(d, k):
    return clean_str(d.get(k, ""))

def make_key(first, last, grad_year):
    return f"{norm_name(first)}|{norm_name(last)}|{clean_str(grad_year)}"

def merge_record(base, incoming, incoming_priority, base_priority):
    """
    Higher priority wins for current fields; older conflicting values get pushed into Previous/Notes.
    """
    cur_fields = {
        "phone","current email","linked in profile address","current residence",
        "current role","current employer name","Type of organization",
        "employment sector or field","graduate studies","grad degree year",
        "IRIS location","IRIS internship","Capstone internship","Thesis title",
        "Advisor","Awards","State when admitted","Admit Term"
    }

    # Fill identity fields if missing
    for f in ["Graduation Year","First Name","Last Name"]:
        if not safe_get(base,f) and safe_get(incoming,f):
            base[f] = incoming[f]

    for f in TARGET_COLS:
        inc = safe_get(incoming, f)
        if not inc:
            continue

        if f in cur_fields:
            base_val = safe_get(base, f)
            if not base_val:
                base[f] = inc
            else:
                if incoming_priority > base_priority and inc != base_val:
                    # push the older value somewhere sensible
                    if f == "current email":
                        base["Previous e-mails"] = append_prev(base.get("Previous e-mails",""), base_val)
                    elif f == "current role":
                        base["work experience highlights/previous positions"] = append_prev(
                            base.get("work experience highlights/previous positions",""),
                            f"Previous role: {base_val}"
                        )
                    else:
                        base["Notes"] = append_prev(base.get("Notes",""), f"Previous {f}: {base_val}")
                    base[f] = inc
        else:
            # non-current: append
            base[f] = append_prev(base.get(f,""), inc)
    return base
soenke = pd.read_csv(paths["soenke"],encoding='latin1')
soenke.head(2)
def parse_soenke(df):
    rows = []
    for _, r in df.iterrows():
        out = empty_target_row()
        out["Graduation Year"] = clean_str(r.get("Class of",""))
        first, last = split_last_first(r.get("Name",""))
        out["First Name"], out["Last Name"] = first, last

        out["current email"] = clean_str(r.get("Personal Email",""))
        out["phone"] = clean_str(r.get("Phone/WhatsApp",""))
        out["linked in profile address"] = clean_str(r.get("LinkedIn",""))
        out["employment sector or field"] = clean_str(r.get("Focus",""))
        out["current residence"] = clean_str(r.get("Base (Feb 2024)",""))
        out["graduate studies"] = clean_str(r.get("Graduate Degree",""))
        out["current role"] = clean_str(r.get("Current Job (Feb 2024)",""))
        out["work experience highlights/previous positions"] = clean_str(r.get("Job Highlights",""))
        out["Awards"] = clean_str(r.get("Awards, Fellowships & Scholarships",""))

        out["IRIS location"] = clean_str(r.get("IRIS",""))
        out["IRIS internship"] = clean_str(r.get("IRIS Internship",""))
        out["Capstone internship"] = clean_str(r.get("Capstone Internship",""))
        out["Thesis title"] = clean_str(r.get("Thesis",""))
        out["Advisor"] = append_prev(clean_str(r.get("Junior Advisor","")), clean_str(r.get("Senior Advisor","")))

        rows.append(out)
    return rows

rows_soenke = parse_soenke(soenke)
display(len(rows_soenke), rows_soenke[0])
linkedin = pd.read_csv(paths["linkedin"],encoding='latin1')
display(linkedin.head(2))
def parse_linkedin_research(df):
    rows = []
    for _, r in df.iterrows():
        out = empty_target_row()
        name = clean_str(r.get("Name",""))
        out["Graduation Year"] = clean_str(r.get("Graduation Year, LIU",""))
        first, last = split_last_first(name)
        out["First Name"], out["Last Name"] = first, last

        out["graduate studies"] = clean_str(r.get("Graduate Degree, Field and Institution",""))
        out["grad degree year"] = clean_str(r.get("Year received graduate degree",""))

        out["current role"] = clean_str(r.get("Current position ","")) or clean_str(r.get("Current position",""))
        out["work experience highlights/previous positions"] = clean_str(r.get("Last position ","")) or clean_str(r.get("Last position",""))
        out["employment sector or field"] = clean_str(r.get("Current field ","")) or clean_str(r.get("Current field",""))
        out["Type of organization"] = clean_str(r.get("Type of organization ","")) or clean_str(r.get("Type of organization",""))
        out["Awards"] = clean_str(r.get("Awards and Recognitions",""))

        contact = clean_str(r.get("Contact info",""))
        out["linked in profile address"] = contact

        emails = extract_emails(contact)
        phones = extract_phones(contact)
        if emails:
            out["current email"] = emails[0]
            if len(emails) > 1:
                out["Previous e-mails"] = "; ".join(emails[1:])
        if phones:
            out["phone"] = phones[0]

        rows.append(out)
    return rows

rows_linkedin = parse_linkedin_research(linkedin)
display(len(rows_linkedin), rows_linkedin[0])
contacts2024 = pd.read_csv(paths["contacts2024"],encoding='latin1')
gathering = pd.read_csv(paths["gathering"],encoding='latin1')
survey = pd.read_csv(paths["survey"], low_memory=False)

display(contacts2024.head(2), gathering.head(2), survey.shape)
def parse_contacts_2024(df):
    rows = []
    for _, r in df.iterrows():
        out = empty_target_row()
        out["Last Name"] = clean_str(r.get("Last name",""))
        out["First Name"] = clean_str(r.get("First name",""))
        out["Graduation Year"] = clean_str(r.get("Year you graduated",""))
        out["current role"] = clean_str(r.get("What is your current professional position or academic studies?",""))
        out["employment sector or field"] = clean_str(r.get("What professional field(s) have you mostly worked in?",""))
        out["Notes"] = clean_str(r.get("Anything else you want to share with our current students?",""))

        blob = clean_str(r.get("Please provide the best way for us/our students to contact you. List all that you wish to share (email, phone, Instagram, Facebook, Linked-In, mailing address, other)..",""))
        emails = extract_emails(blob)
        phones = extract_phones(blob)
        if emails:
            out["current email"] = emails[0]
            if len(emails) > 1:
                out["Previous e-mails"] = "; ".join(emails[1:])
        if phones:
            out["phone"] = phones[0]
        if "linkedin" in blob.lower() and not out["linked in profile address"]:
            out["linked in profile address"] = blob

        rows.append(out)
    return rows

def parse_gathering(df):
    rows = []
    for _, r in df.iterrows():
        out = empty_target_row()
        name = clean_str(r.get("What is your name (first, last)",""))
        parts = re.split(r"\s+", name.strip())
        if len(parts) >= 2:
            out["First Name"] = parts[0].strip().strip(",")
            out["Last Name"] = parts[-1].strip().strip(",")
        else:
            out["First Name"] = name

        out["current email"] = clean_str(r.get("Your email address to send event reminder:",""))
        out["Notes"] = clean_str(r.get("Please list any dietary restrictions or allergies you may have.",""))
        rows.append(out)
    return rows

def parse_survey(df):
    rows = []
    for _, r in df.iterrows():
        out = empty_target_row()
        out["First Name"] = clean_str(r.get("What is your first name?",""))
        out["Last Name"] = clean_str(r.get("What is your last name?",""))
        out["Graduation Year"] = clean_str(r.get("Graduation Year, LIU",""))

        # required per instructions: indicate they responded
        out["Responded to 2020 Alumni Survey?"] = "Yes"

        out["current residence"] = clean_str(r.get("What city/state/country are you currently based in?",""))
        out["current role"] = clean_str(r.get("What is your current professional title/position?",""))
        out["current employer name"] = clean_str(r.get("Please list your current employer (optional)",""))
        out["employment sector or field"] = clean_str(r.get("What professional field(s) have you mostly worked in (select all that apply)?",""))

        out["work experience highlights/previous positions"] = clean_str(
            r.get("Please provide brief highlights of your career path to date and how they have been influenced by your FW/GC/LIU Global education?","")
        )
        out["linked in profile address"] = clean_str(r.get("Please provide your LinkedIn Profile address if you have one (optional).",""))

        blob = clean_str(r.get("Please provide the best way for us to contact you. We promise not to give out this information publicly. List all that you wish to share (email, phone, Instagram, Facebook, mailing address, other).",""))
        emails = extract_emails(blob)
        phones = extract_phones(blob)
        if emails:
            out["current email"] = emails[0]
            if len(emails) > 1:
                out["Previous e-mails"] = "; ".join(emails[1:])
        if phones:
            out["phone"] = phones[0]

        rows.append(out)
    return rows

rows_contacts2024 = parse_contacts_2024(contacts2024)
rows_gathering = parse_gathering(gathering)
rows_survey = parse_survey(survey)

print(len(rows_contacts2024), len(rows_gathering), len(rows_survey))
from docx import Document

def docx_paras(path):
    doc = Document(path)
    return [p.text.strip() for p in doc.paragraphs if p.text and p.text.strip()]

# Emails docx
emails_paras = docx_paras(paths["docx_emails"])

def parse_emails_docx(paras):
    rows = []
    current_year = ""
    for line in paras:
        m = re.search(r"May\s+(20\d{2})\s+grads", line, re.IGNORECASE)
        if m:
            current_year = m.group(1)
            continue
        for e in extract_emails(line):
            out = empty_target_row()
            out["Graduation Year"] = current_year
            out["current email"] = e
            out["Notes"] = "From Emails alums 2020-2024 grads.docx"
            rows.append(out)
    return rows

rows_emails_docx = parse_emails_docx(emails_paras)

# Teresa bullets docx
teresa_paras = docx_paras(paths["docx_teresa"])

def parse_teresa_docx(paras):
    rows = []
    for line in paras:
        line2 = line.lstrip("-").strip()
        m = re.match(r"(.+?)\s*\((?:Class of|class of)\s*(\d{4})\)\s*,\s*(.+)$", line2)
        if not m:
            continue
        name, year, desc = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        parts = name.split()
        first = parts[0] if parts else name
        last = parts[-1] if len(parts) >= 2 else ""
        out = empty_target_row()
        out["First Name"], out["Last Name"] = first, last
        out["Graduation Year"] = year
        out["work experience highlights/previous positions"] = desc
        out["Notes"] = "From LIST OF ALUMNI from Teresa.docx"
        rows.append(out)
    return rows

rows_teresa = parse_teresa_docx(teresa_paras)

# 5-year table docx
doc_5yrs = Document(paths["docx_5yrs"])

def parse_5yrs_docx(doc):
    rows = []
    for t in doc.tables:
        headers = [c.text.strip() for c in t.rows[0].cells]

        def idx(col):
            for i,h in enumerate(headers):
                if h.strip().lower() == col.lower():
                    return i
            return None

        i_class = idx("Class")
        i_last  = idx("Last name")
        i_first = idx("First name")
        i_2012  = idx("2012")
        i_2013  = idx("2013")

        if i_class is None or i_last is None or i_first is None:
            continue

        for rr in t.rows[1:]:
            cells = [c.text.strip() for c in rr.cells]
            if not any(cells):
                continue

            out = empty_target_row()
            out["Graduation Year"] = cells[i_class] if i_class < len(cells) else ""
            out["Last Name"] = cells[i_last] if i_last < len(cells) else ""
            out["First Name"] = cells[i_first] if i_first < len(cells) else ""

            notes = []
            if i_2012 is not None and i_2012 < len(cells) and cells[i_2012]:
                notes.append(f"2012: {cells[i_2012]}")
            if i_2013 is not None and i_2013 < len(cells) and cells[i_2013]:
                notes.append(f"2013: {cells[i_2013]}")

            if notes:
                out["work experience highlights/previous positions"] = " | ".join(notes)
                out["Notes"] = "From ALumni- 5 yrs.docx"
            rows.append(out)
    return rows

rows_5yrs = parse_5yrs_docx(doc_5yrs)

print(len(rows_emails_docx), len(rows_teresa), len(rows_5yrs))
sources = [
    ("soenke", rows_soenke, 100),
    ("linkedin", rows_linkedin, 90),
    ("contacts2024", rows_contacts2024, 60),
    ("gathering", rows_gathering, 50),
    ("emails_docx", rows_emails_docx, 40),
    ("survey2020", rows_survey, 20),
    ("teresa", rows_teresa, 10),
    ("alumni5yrs", rows_5yrs, 5),
]

master = {}          # key -> record
master_priority = {} # key -> priority currently defining record

def add_rows(rows, priority):
    for rec in rows:
        # primary key: name + grad year; if missing year, use UNKNOWN
        gy = clean_str(rec.get("Graduation Year","")) or "UNKNOWN"
        k = make_key(rec.get("First Name",""), rec.get("Last Name",""), gy)

        if k not in master:
            master[k] = rec.copy()
            master_priority[k] = priority
        else:
            base_pr = master_priority[k]
            master[k] = merge_record(master[k], rec, incoming_priority=priority, base_priority=base_pr)
            master_priority[k] = max(base_pr, priority)

for _, rows, pr in sources:
    add_rows(rows, pr)

len(master)
def key_parts(k):
    a,b,c = k.split("|")
    return a,b,c

def merge_unknown_year(master, master_priority):
    keys = list(master.keys())
    to_delete = set()
    merged = 0
    for k in keys:
        fi, la, yi = key_parts(k)
        if yi != "UNKNOWN":
            continue
        # find a known-year record with same normalized name
        for kk in keys:
            fj, lb, yj = key_parts(kk)
            if yj == "UNKNOWN":
                continue
            if fi == fj and la == lb:
                # merge UNKNOWN into known-year
                pr_u = master_priority[k]
                pr_k = master_priority[kk]
                master[kk] = merge_record(master[kk], master[k], incoming_priority=pr_u, base_priority=pr_k)
                to_delete.add(k)
                merged += 1
                break
    for k in to_delete:
        master.pop(k, None)
        master_priority.pop(k, None)
    return merged

merged_n = merge_unknown_year(master, master_priority)
print("Merged UNKNOWN-year into known-year:", merged_n)
print("Total records:", len(master))
final_df = pd.DataFrame(list(master.values()))
final_df = final_df[TARGET_COLS].copy()

# sort
final_df["_gy"] = pd.to_numeric(final_df["Graduation Year"], errors="coerce")
final_df = final_df.sort_values(["_gy","Last Name","First Name"], na_position="last").drop(columns=["_gy"])

out_main = "/content/Feb_2026_LIU_Global_Alumni_Data_Research_FILLED.xlsx"
final_df.to_excel(out_main, index=False)

out_main, final_df.shape

All files found.


229

{'Admit Term': '',
 'Graduation Year': '2025',
 'First Name': 'Amanda',
 'Last Name': 'James',
 'State when admitted': '',
 'phone': '1(718)663-9173',
 'current email': 'adajames1128@gmail.com',
 'Previous e-mails': '',
 'linked in profile address': '',
 'current residence': '',
 'current role': '',
 'current employer name': '',
 'Type of organization': '',
 'employment sector or field ': '',
 'work experience highlights/previous positions': '',
 'graduate studies': '',
 'grad degree year': '',
 'Responded to 2020 Alumni Survey?': '',
 'IRIS location': '',
 'IRIS internship': '',
 'Capstone internship': '',
 'Thesis title': '',
 'Advisor': 'Louise/Nigel',
 'Awards': '',
 'Notes': '',
 'employment sector or field': ''}

,Name,"Graduation Year, LIU",Undergraduate Studies,"Graduate Degree, Field and Institution",Year received graduate degree,Level,Last position,Current position,Current field,Type of organization,Awards and Recognitions,Contact info
0,Krystal Cerisier,2021,\n\nBachelors at Long Island University,Masters in Community Organization and Advocacy...,2022,"Bachelors ,Masters",Homeless Union Organizer at Vocal NY,Research Fellowship at Leadership For Democrac...,Social Work,Private,NaN,linkedin.com/in/krystal-cerisier-bbb3a0b5
1,Sara Crouch,2021,"Bachelors at Long Island University, Global St...",NaN,NaN,Bachelors,Peace Education Intern at Creative Response to...,Restorative Practices Coordinator at Denver Pu...,Education,Public,NaN,linkedin.com/in/sara-crouch-a79505151


85

...

,ï»¿Last name,First name,Year you graduated,What is your current professional position or academic studies?,What professional field(s) have you mostly worked in?,Do you give LIU Global permission to share your contact information internally for members of the LIU Global community?,"Please provide the best way for us/our students to contact you. List all that you wish to share (email, phone, Instagram, Facebook, Linked-In, mailing address, other)..",Anything else you want to share with our current students?
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tuai,Alex,2015.0,"ESG Metrics and Regulations Associate, Novata\...","International diplomacy, academia, ESG/sustain...",Yes,Alex.w.tuai@gmail.com\nLinkedIn.com/alextuai\n...,NaN


,LIU Global Informal Gathering in Brooklyn,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Timestamp,Can you attend?,"What is your name (first, last)",Your email address to send event reminder:,Please list any dietary restrictions or allerg...
1,2025/05/29 12:26:09 PM CST,"Yes, I'll be there",Alexandra Martinez,alexandra72111@gmail.com,NaN


(240, 38)

13 8 240
78 15 89
Merged UNKNOWN-year into known-year: 39
Total records: 461


('/content/Feb_2026_LIU_Global_Alumni_Data_Research_FILLED.xlsx', (461, 25))